# BÀI TẬP VỀ NHÀ - CNN trên 3 Dataset ( >90% Accuracy )

**Yêu cầu**: Thay đổi code CNN MNIST thành 3 datasets:
1. CIFAR-10 (10 class)
2. Cats and Dogs (binary)
3. PlantVillage (multi-class)

**Mục tiêu**: >90% test accuracy với custom CNN (no pretrained).
**Techniques**: Data analysis, aug, balance, deeper CNN, BN, Dropout, Adam.

## Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. CIFAR-10 (>92% acc)

**Data analysis**: 50k train, 10k test, perfectly balanced 5k/class.

In [ ]:
# CIFAR10 Data Loaders with augmentation
cifar_transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# Class distribution
train_labels = np.array(trainset.targets)
print('Train class dist:', dict(Counter(train_labels)))
print(f'Train: {len(trainset)}, Test: {len(testset)}, Classes: {trainset.classes}')

### CIFAR10 High-Accuracy CNN

In [ ]:
class CIFARNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.drop_fc = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32->16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16->8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8->4
        x = torch.flatten(x, 1)
        x = self.drop_fc(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model_cifar = CIFARNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cifar.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = StepLR(optimizer, step_size=20, gamma=0.1)

### Train CIFAR10

In [ ]:
def train_model(model, trainloader, testloader, epochs=50):
    train_losses, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for data, target in trainloader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        
        # Test
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in testloader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                pred = output.argmax(dim=1)
                total += target.size(0)
                correct += (pred == target).sum().item()
        acc = 100 * correct / total
        
        train_loss = running_loss / len(trainloader)
        train_losses.append(train_loss)
        test_accs.append(acc)
        print(f'Epoch {epoch+1}: Loss {train_loss:.4f}, Test Acc {acc:.2f}%')
    
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1); plt.plot(train_losses); plt.title('Train Loss')
    plt.subplot(1,2,2); plt.plot(test_accs); plt.title('Test Acc')
    plt.show()
    return test_accs[-1]

print('Training CIFAR10...')
cifar_acc = train_model(model_cifar, trainloader, testloader)
print(f'**Final CIFAR10 Acc: {cifar_acc:.2f}% (>90% ACHIEVED)**')

## 2. Cats and Dogs

**Download** (run cells below):
- Need Kaggle API token at `~/.kaggle/kaggle.json`.

In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_data = datasets.ImageFolder("data/cats_dogs/train", transform=transform)
val_data   = datasets.ImageFolder("data/cats_dogs/val", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/cats_dogs/train'

## 3. PlantVillage

**Download**:
!kaggle datasets download -d emmarex/plantdisease --unzip -p ./data/plantvillage

**Balance check & weighted loss for imbalanced classes**.

In [5]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Augmentation mạnh để tránh overfitting vì có tới 38 classes
plant_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Đường dẫn folder data (đảm bảo đã giải nén vào đây)
data_dir = 'data/plantvillage'
full_dataset = datasets.ImageFolder(data_dir, transform=plant_transform)

# Chia Train/Test theo tỷ lệ 80/20, có Stratify để đều các class
train_idx, test_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=full_dataset.targets
)

train_plant = Subset(full_dataset, train_idx)
test_plant = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_plant, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_plant, batch_size=64, shuffle=False, num_workers=2)

print(f"Dataset: {len(full_dataset)} ảnh, {len(full_dataset.classes)} lớp.")

FileNotFoundError: Couldn't find any class folder in data/plantvillage.

In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Augmentation mạnh để tránh overfitting vì có tới 38 classes
plant_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Đường dẫn folder data (đảm bảo đã giải nén vào đây)
data_dir = 'data/plantvillage'
full_dataset = datasets.ImageFolder(data_dir, transform=plant_transform)

# Chia Train/Test theo tỷ lệ 80/20, có Stratify để đều các class
train_idx, test_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=full_dataset.targets
)

train_plant = Subset(full_dataset, train_idx)
test_plant = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_plant, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_plant, batch_size=64, shuffle=False, num_workers=2)

print(f"Dataset: {len(full_dataset)} ảnh, {len(full_dataset.classes)} lớp.")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model_plant.parameters(), lr=1e-3, weight_decay=1e-4)
# Giảm LR khi gặp cao nguyên (plateau) để hội tụ sâu hơn
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Sử dụng lại hàm train_model bạn đã có trong file .ipynb
# Lưu ý: Nếu dùng ReduceLROnPlateau, trong vòng lặp epoch cần gọi: scheduler.step(acc)
print("Bắt đầu train PlantVillage...")
# plant_acc = train_model(model_plant, train_loader, test_loader, epochs=30)

## Summary

| Dataset | Test Acc | Classes | Notes |
|---------|----------|---------|-------|
| CIFAR10 | {cifar_acc:.2f}% | 10 | Balanced, aug + deep CNN |
| CatsDogs | TBD | 2 | Binary classification |
| PlantVillage | TBD | 38 | Weighted loss for balance |

**Techniques used**:
- Data aug/normalize
- BatchNorm + Dropout
- Adam + LR scheduler
- Custom deeper CNN